# Phase 3 — Dataset Construction

Reads `weekly_features.parquet` (50K users × 4 weeks), computes churn labels,
and produces model-ready datasets:
- **XGBoost**: single row per user (aggregated features)
- **LSTM**: `(users, 4, features)` sequential tensor

**Prerequisite**: run `phase2_feature_engineering.ipynb` first.

In [ ]:
import pandas as pd
import numpy as np
import datetime

# Temporal constants
DATA_START = datetime.date(2024, 1, 1)
OBS_START  = datetime.date(2024, 1, 29)
OBS_END    = datetime.date(2024, 2, 26)
PRED_START = datetime.date(2024, 2, 26)  # week 9
PRED_END   = datetime.date(2024, 3, 11)  # week 10 end

# Load data
wf = pd.read_parquet("../data/processed/weekly_features.parquet")
users = pd.read_parquet("../data/raw/users.parquet", columns=["user_id"])
sessions = pd.read_parquet("../data/raw/session_logs.parquet", columns=["user_id", "start_time"])

print(f"weekly_features: {wf.shape}")
print(f"users: {len(users):,}")
print(f"sessions: {len(sessions):,}")

## 1. Churn Labels

Prediction window = weeks 9–10 (2024-02-26 to 2024-03-11).  
**churn = 1** if a user has zero sessions in the prediction window.

In [ ]:
pred_sessions = sessions[
    (sessions["start_time"].dt.date >= PRED_START) &
    (sessions["start_time"].dt.date < PRED_END)
]
active_in_pred = set(pred_sessions["user_id"].unique())

labels = users[["user_id"]].copy()
labels["churn"] = (~labels["user_id"].isin(active_in_pred)).astype(int)

churn_rate = labels["churn"].mean()
print(f"Churn rate: {churn_rate:.1%} ({labels['churn'].sum():,} / {len(labels):,})")
print(f"Expected: ~15% from about_to_churn persona + some casual users")

## 2. Feature Column Classification

In [ ]:
# Category A: mean across 4 weeks (behavioral averages)
mean_cols = [
    "weekly_session_count", "avg_session_duration_min", "total_playtime_min",
    "peak_hour_ratio", "weekend_ratio", "session_regularity",
    "weekly_session_count_norm", "total_playtime_min_norm",
    "avg_latency", "avg_fps", "frame_drop_rate", "disconnect_rate",
    "avg_bitrate", "avg_jitter", "packet_loss_avg", "crash_exit_ratio",
    "unique_games_played", "top_game_concentration", "genre_entropy",
    "daily_playtime_std", "daily_playtime_cv", "session_duration_std",
]

# Category B: last week (week 4) value (recency signal)
last_cols = [
    "session_count_wow_change", "playtime_wow_change",
    "session_count_vs_baseline", "playtime_vs_baseline",
    "longest_inactive_days", "new_game_trial_rate",
]

# Extra last-week variants for key features (mean + last gives level + recency)
extra_last_cols = [
    "weekly_session_count", "total_playtime_min", "crash_exit_ratio",
]

# Category C: static / user-level (same all 4 weeks)
static_cols = [
    "current_tier_numeric", "days_since_signup",
    "tier_changes_count", "has_downgraded",
    "total_spend_last_4w", "payment_count_last_4w",
    "failed_payment_count", "refund_count",
    "days_since_last_payment", "payment_frequency_change",
]

print(f"Mean features: {len(mean_cols)}")
print(f"Last-week features: {len(last_cols)}")
print(f"Extra last-week variants: {len(extra_last_cols)}")
print(f"Static features: {len(static_cols)}")
print(f"+ 1 slope feature (computed below)")

## 3. XGBoost Aggregation

In [ ]:
# Category A: mean across 4 weeks (pandas skips NaN by default)
agg_mean = wf.groupby("user_id")[mean_cols].mean()

# Category B: last week (week 4)
week4 = wf[wf["week_num"] == 4].set_index("user_id")
agg_last = week4[last_cols].rename(columns={c: f"{c}_last" for c in last_cols})

# Extra last-week variants
agg_extra_last = week4[extra_last_cols].rename(columns={c: f"{c}_last" for c in extra_last_cols})

# Category C: static (take first row per user)
agg_static = wf.drop_duplicates("user_id").set_index("user_id")[static_cols]

# Category D: playtime slope across 4 weeks
def compute_slope(group):
    x = group["week_num"].values.astype(float)
    y = group["total_playtime_min"].values.astype(float)
    n = len(x)
    slope = (n * np.dot(x, y) - x.sum() * y.sum()) / (n * np.dot(x, x) - x.sum() ** 2)
    return slope

agg_slope = wf.groupby("user_id").apply(compute_slope, include_groups=False)
agg_slope.name = "playtime_slope"

# Combine all
xgb_df = agg_mean.join(agg_last).join(agg_extra_last).join(agg_static).join(agg_slope)
xgb_df = xgb_df.join(labels.set_index("user_id"))

print(f"XGBoost dataset: {xgb_df.shape}")
print(f"Columns ({len(xgb_df.columns)}): {list(xgb_df.columns)}")

In [ ]:
# Quick sanity check
assert len(xgb_df) == len(users), f"Expected {len(users)} rows, got {len(xgb_df)}"
assert "churn" in xgb_df.columns
assert xgb_df["churn"].isin([0, 1]).all()

# NaN counts — XGBoost handles NaN natively, but review what's null
null_counts = xgb_df.isnull().sum()
print("Columns with NaN (XGBoost handles these natively):")
print(null_counts[null_counts > 0].to_string())

print(f"\nChurn distribution:\n{xgb_df['churn'].value_counts().to_string()}")
print(f"\nPASS: XGBoost dataset shape and labels OK")

## 4. LSTM Sequential Format

In [ ]:
# Feature columns for LSTM (all weekly features, excluding keys and static columns)
# Static columns are the same across all 4 weeks — include them so LSTM
# has the same feature set as XGBoost, just in sequential form.
lstm_feature_cols = [c for c in wf.columns if c not in ["user_id", "week_num"]]

# Fill nulls for tensor compatibility
lstm_df = wf.copy()

# wow_change, new_game_trial_rate: 0 = no change (neutral)
lstm_df[["session_count_wow_change", "playtime_wow_change", "new_game_trial_rate"]] = \
    lstm_df[["session_count_wow_change", "playtime_wow_change", "new_game_trial_rate"]].fillna(0.0)

# vs_baseline: 1.0 = same as baseline (neutral)
lstm_df[["session_count_vs_baseline", "playtime_vs_baseline"]] = \
    lstm_df[["session_count_vs_baseline", "playtime_vs_baseline"]].fillna(1.0)

# days_since_last_payment: -1 = never paid (sentinel)
lstm_df["days_since_last_payment"] = lstm_df["days_since_last_payment"].fillna(-1)

# payment_frequency_change: 0 = no change (neutral)
lstm_df["payment_frequency_change"] = lstm_df["payment_frequency_change"].fillna(0.0)

# Remaining nulls (session_duration_std, daily_playtime_cv from low-session weeks)
lstm_df[lstm_feature_cols] = lstm_df[lstm_feature_cols].fillna(0.0)

# Verify no NaN remains
remaining_nulls = lstm_df[lstm_feature_cols].isnull().sum().sum()
assert remaining_nulls == 0, f"{remaining_nulls} NaN values remain in LSTM features"
print(f"LSTM features: {len(lstm_feature_cols)} columns, all NaN resolved")
print(f"Null fill strategy: wow/trial→0, vs_baseline→1, days_since_payment→-1, rest→0")

In [ ]:
# Reshape to 3D tensor: (users, 4 weeks, features)
lstm_df = lstm_df.sort_values(["user_id", "week_num"])

user_ids = lstm_df["user_id"].unique()
n_users = len(user_ids)
n_weeks = 4
n_features = len(lstm_feature_cols)

X_lstm = lstm_df[lstm_feature_cols].values.reshape(n_users, n_weeks, n_features)
y_lstm = labels.set_index("user_id").loc[user_ids, "churn"].values

print(f"X shape: {X_lstm.shape}  (users × weeks × features)")
print(f"y shape: {y_lstm.shape}")
print(f"Feature names: {lstm_feature_cols}")

assert X_lstm.shape == (n_users, 4, n_features)
assert not np.isnan(X_lstm).any(), "NaN found in LSTM tensor"
print(f"PASS: LSTM tensor OK")

## 5. Stratified Train/Val/Test Split

Stratify by the `churn` label so train/val/test all reflect the ~12% churn rate.
The same `user_id` partition is applied to both XGBoost and LSTM formats so a
user appearing in the XGBoost train set also appears in the LSTM train set.

Ratio: 70% train / 15% val / 15% test. Seed = 42.

In [ ]:
SEED = 42
TRAIN_RATIO, VAL_RATIO = 0.70, 0.15  # test_ratio = 1 - TRAIN_RATIO - VAL_RATIO

rng = np.random.default_rng(SEED)

# Stratify by churn label: shuffle each class independently then slice
labels_df = labels.set_index("user_id").loc[user_ids].reset_index()
pos_ids = labels_df.loc[labels_df["churn"] == 1, "user_id"].to_numpy()
neg_ids = labels_df.loc[labels_df["churn"] == 0, "user_id"].to_numpy()

def stratified_slice(ids):
    perm = rng.permutation(len(ids))
    n_train = int(len(ids) * TRAIN_RATIO)
    n_val   = int(len(ids) * VAL_RATIO)
    return ids[perm[:n_train]], ids[perm[n_train:n_train+n_val]], ids[perm[n_train+n_val:]]

pos_train, pos_val, pos_test = stratified_slice(pos_ids)
neg_train, neg_val, neg_test = stratified_slice(neg_ids)

train_ids = np.concatenate([pos_train, neg_train])
val_ids   = np.concatenate([pos_val,   neg_val])
test_ids  = np.concatenate([pos_test,  neg_test])
rng.shuffle(train_ids); rng.shuffle(val_ids); rng.shuffle(test_ids)

splits = {"train": train_ids, "val": val_ids, "test": test_ids}

print(f"{'split':<6} {'users':>7} {'churn':>7} {'rate':>7}")
for name, ids in splits.items():
    churn_n = labels_df.set_index("user_id").loc[ids, "churn"].sum()
    print(f"{name:<6} {len(ids):>7,} {churn_n:>7,} {churn_n/len(ids):>7.1%}")

assert len(train_ids) + len(val_ids) + len(test_ids) == len(user_ids)
assert set(train_ids) | set(val_ids) | set(test_ids) == set(user_ids)
assert set(train_ids) & set(val_ids) == set()
assert set(train_ids) & set(test_ids) == set()
assert set(val_ids) & set(test_ids) == set()
print("\nPASS: split is exhaustive and disjoint")

## 6. Save Outputs

Six files: train/val/test × XGBoost/LSTM. Path convention:
`xgboost_<split>.parquet` and `lstm_<split>.npz`.

In [ ]:
import os
os.makedirs("../data/processed", exist_ok=True)

# Map user_id -> row position in the LSTM tensor (rows are ordered by user_ids)
uid_to_row = {uid: i for i, uid in enumerate(user_ids)}

for split_name, split_ids in splits.items():
    split_id_set = set(split_ids)

    # XGBoost: filter rows
    xgb_split = xgb_df.loc[xgb_df.index.isin(split_id_set)]
    xgb_split.to_parquet(f"../data/processed/xgboost_{split_name}.parquet")

    # LSTM: select rows from the tensor by user_id position
    row_idx = np.array([uid_to_row[uid] for uid in split_ids])
    np.savez(
        f"../data/processed/lstm_{split_name}.npz",
        X=X_lstm[row_idx],
        y=y_lstm[row_idx],
        user_ids=np.array(split_ids),
        feature_names=np.array(lstm_feature_cols),
    )
    print(f"{split_name:<6} xgb={xgb_split.shape} | lstm X={X_lstm[row_idx].shape}")

## 7. Summary

In [ ]:
print("=" * 55)
print("Dataset Construction Summary")
print("=" * 55)
print(f"Users:          {n_users:,}")
print(f"Churn rate:     {churn_rate:.1%} ({labels['churn'].sum():,} churned)")
print()
print(f"XGBoost cols:   {xgb_df.shape[1]}")
print(f"  Mean features:       {len(mean_cols)}")
print(f"  Last-week features:  {len(last_cols) + len(extra_last_cols)}")
print(f"  Static features:     {len(static_cols)}")
print(f"  Slope features:      1")
print(f"  Label:               1")
print()
print(f"LSTM shape:     ({n_users}, 4, {n_features})")
print()
print(f"Splits (stratified by churn, seed={SEED}):")
for name, ids in splits.items():
    cn = labels_df.set_index('user_id').loc[ids, 'churn'].sum()
    print(f"  {name:<5} {len(ids):>6,} users  churn={cn:>5,} ({cn/len(ids):.1%})")
print()
print("Saved to: data/processed/")
print("  xgboost_{train,val,test}.parquet")
print("  lstm_{train,val,test}.npz")
print("=" * 55)